# Implementation of Sparse 3D point Reconstruction Pipeline
- Utilizes COLMAP Framework 
- Outputs submissiong.csv for Kaggle and sparse 3D point cloud for 3D reconstruction
- This notebook requires a COLMAP installation: Requires COLMAP >= 3.7 installed system-wide

In [ ]:
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import os
import json
import struct
from tqdm.auto import tqdm

In [ ]:
#check that colmap works
!colmap feature_extractor -h

In [ ]:
#Forcing COLMAP to run headless

os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.environ["DISPLAY"] = ""
os.environ["XDG_RUNTIME_DIR"] = "/tmp"

ENV = os.environ

## Setup
- Data loading
- COLMAP setup

In [ ]:
#constants
SCENE_STEPS = [
    "prepare_images",
    "feature_extraction",
    "matching",
    "mapping",
    "pose_extraction",
]

DATA_ROOT = Path("./data/train")
CACHE_ROOT = Path("./cache")
SFM_ROOT = CACHE_ROOT / "sfm"
SFM_ROOT.mkdir(parents=True, exist_ok=True)

CLUSTERS_CSV = CACHE_ROOT / "clusters.csv"
SUBMISSION_CSV = Path("submission.csv")

COLMAP_BIN = "colmap"
CAMERA_MODEL = "SIMPLE_RADIAL"
MAX_IMAGE_SIZE = 2000

In [ ]:
#utils
def run(cmd, cwd=None, timeout=900):
    print(" ".join(map(str, cmd)))
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        env=ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError("COLMAP command failed")


def qvec2rotmat(qvec):
    w, x, y, z = qvec
    return np.array([
        [1 - 2*y*y - 2*z*z, 2*x*y - 2*z*w,     2*x*z + 2*y*w],
        [2*x*y + 2*z*w,     1 - 2*x*x - 2*z*z, 2*y*z - 2*x*w],
        [2*x*z - 2*y*w,     2*y*z + 2*x*w,     1 - 2*x*x - 2*y*y]
    ])


def load_global_image_list(dataset):
    images = []
    for p in (DATA_ROOT / dataset).rglob("*"):
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            images.append(p.name)
    return sorted(images)


## SfM Pipeline: Reconstruction logic

In [ ]:
def reconstruct_scene(dataset, scene, image_ids, global_image_names, global_name_to_idx, pbar=None):

    scene_dir = SFM_ROOT / dataset / scene
    img_dir = scene_dir / "images"
    db_path = scene_dir / "database.db"
    sparse_dir = scene_dir / "sparse"

    pairs_npz = CACHE_ROOT / dataset / "pairs_topk_K30.npz"
    pairs_txt = scene_dir / "pairs.txt"

    # reset scene
    if scene_dir.exists():
        shutil.rmtree(scene_dir)

    scene_dir.mkdir(parents=True)
    img_dir.mkdir()

    if pbar:
        pbar.update(1)


    # Copy images

    for img_id in image_ids:
        matches = list((DATA_ROOT / dataset).rglob(f"{img_id}.*"))
        if not matches:
            raise RuntimeError(f"Image not found: {img_id}")
        shutil.copy(matches[0], img_dir / matches[0].name)

    if len(list(img_dir.iterdir())) == 0:
        raise RuntimeError("No images copied")


    # Write pairs.txt from (a, b) edge list

    if not pairs_npz.exists():
        raise RuntimeError("pairs_topk_K30.npz not found")

    data = np.load(pairs_npz)
    a = data["a"]
    b = data["b"]

    cluster_image_names = sorted(p.name for p in img_dir.iterdir())
    cluster_indices = {
        global_name_to_idx[name]
        for name in cluster_image_names
        if name in global_name_to_idx
    }

    num_pairs = 0
    with open(pairs_txt, "w") as f:
        for i, j in zip(a, b):
            if i in cluster_indices and j in cluster_indices:
                f.write(f"{global_image_names[i]} {global_image_names[j]}\n")
                num_pairs += 1

    print("Pairs written:", num_pairs)

    if num_pairs < 5:
        raise RuntimeError("Too few pairs for reliable mapping")


    # Feature extraction

    run([
        COLMAP_BIN, "feature_extractor",
        "--database_path", db_path,
        "--image_path", img_dir,
        "--ImageReader.camera_model", CAMERA_MODEL,
        "--ImageReader.single_camera", "1",
        "--SiftExtraction.use_gpu", "0",
        "--SiftExtraction.max_image_size", str(MAX_IMAGE_SIZE),
        "--SiftExtraction.num_threads", "4",
        "--SiftExtraction.max_num_features", "4096",
        "--SiftExtraction.first_octave", "0",
    ])

    if pbar:
        pbar.update(1)

    # Matching (Importing pairs)


    run([
        COLMAP_BIN, "matches_importer",
        "--database_path", db_path,
        "--match_list_path", pairs_txt,
        "--SiftMatching.use_gpu", "0",
    ])

    if pbar:
        pbar.update(1)

    # Mapping

    sparse_dir.mkdir(parents=True, exist_ok=True)

    run([
        COLMAP_BIN, "mapper",
        "--database_path", db_path,
        "--image_path", img_dir,
        "--output_path", sparse_dir,
        "--Mapper.min_num_matches", "8",
        "--Mapper.init_num_trials", "50",
        "--Mapper.max_reg_trials", "2",
        "--Mapper.ba_global_max_num_iterations", "5",
        "--Mapper.ba_global_max_refinements", "1",
    ], timeout=3600)

    if pbar:
        pbar.update(1)

    model_dir = sparse_dir / "0"
    if not model_dir.exists():
        return None

    return model_dir


## SfM per Scene + Pose extraction

In [ ]:
def extract_poses(model_dir, scene_dir, dataset, scene):
    txt_dir = scene_dir / "sparse_txt"
    if txt_dir.exists():
        shutil.rmtree(txt_dir)
    txt_dir.mkdir()

    run([
        COLMAP_BIN, "model_converter",
        "--input_path", model_dir,
        "--output_path", txt_dir,
        "--output_type", "TXT",
    ])

    poses = []
    skip_next = False  # skip 2D observation lines

    with open(txt_dir / "images.txt", "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            if skip_next:
                # ignore 2d points
                skip_next = False
                continue

            parts = line.split()

            # Header line must have at least 10 tokens:
            # IMAGE_ID QW QX QY QZ TX TY TZ CAMERA_ID NAME
            if len(parts) < 10:
                continue

      
            try:
                _img_id_int = int(parts[0])
            except ValueError:
                continue

            name = parts[9]
            if not any(name.lower().endswith(ext) for ext in [".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"]):
                continue

            _, qw, qx, qy, qz, tx, ty, tz, _, name = parts[:10]

            qvec = np.array([float(qw), float(qx), float(qy), float(qz)])
            tvec = np.array([float(tx), float(ty), float(tz)])

            R = qvec2rotmat(qvec)
            C = -R.T @ tvec

            poses.append({
                "dataset": dataset,
                "scene": scene,
                "image": name,
                "image_id": Path(name).stem,
                "rotation_matrix": ";".join(map(str, R.T.flatten())),
                "translation_vector": ";".join(map(str, C)),
            })

            # After every header line, COLMAP writes one 2D observation line
            skip_next = True

    return pd.DataFrame(poses)


# Run SfM Pipeline per Scene: Main Loop

In [ ]:
clusters_df = pd.read_csv(CLUSTERS_CSV)
clusters_df = clusters_df[clusters_df.scene != "outliers"]

groups = list(clusters_df.groupby(["dataset", "scene"]))
all_rows = []

current_dataset = None
global_image_names = None
global_name_to_idx = None

for i, ((dataset, scene), g) in enumerate(groups, 1):

    image_ids = g.image_id.tolist()
    print(f"\n[{i}/{len(groups)}] {dataset} / {scene}")

    if len(image_ids) < 2:
        print("Skipping: too few images")
        continue

    if dataset != current_dataset:
        global_image_names = load_global_image_list(dataset)
        global_name_to_idx = {n: i for i, n in enumerate(global_image_names)}
        current_dataset = dataset

    with tqdm(total=len(SCENE_STEPS), desc=f"{dataset}/{scene}", leave=False) as pbar:
        try:
            model_dir = reconstruct_scene(
                dataset,
                scene,
                image_ids,
                global_image_names,
                global_name_to_idx,
                pbar,
            )

            if model_dir is None:
                print("Reconstruction failed")
                continue

            scene_dir = SFM_ROOT / dataset / scene
            poses_df = extract_poses(model_dir, scene_dir, dataset, scene)

            all_rows.append(poses_df)
            pbar.update(1)

            print(f"Finished {dataset}/{scene}")

        except Exception as e:
            print(f"Skipping {dataset}/{scene} due to error:")
            print(e)

# Output

In [ ]:
if all_rows:
    submission_df = pd.concat(all_rows, ignore_index=True)
    submission_df.to_csv(SUBMISSION_CSV, index=False)
    print("Saved submission.csv")
    display(submission_df.head())
else:
    print("No valid reconstructions")